In [ ]:
from os import listdir as ls
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import arviz as az

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import OUTPUTS_PATH, OXCGRT_LOCATION_CMAP, OXCGRT_COLMAP, MOB_LOCATION_NAME_MAP

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

job_path = OUTPUTS_PATH / "48930936"
all_countries = [iso3 for iso3 in ls(job_path) if "oxcgrt" in ls(job_path / iso3)]
records = []
for iso3 in all_countries:
    analysis_path = job_path / iso3 / "oxcgrt"
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    medians = idata.posterior["mob_weights"].median(dim=("chain", "draw"))
    floor = float(idata.posterior["scale_floor"].median(dim=("chain", "draw")))
    best_policy = int(medians.argmax().item())
    total_weight = float(np.sum(medians))
    scale_factor = (1.0 - floor) / total_weight
    row = {
        "ISO_A3": iso3,
        "best_policy": best_policy,
        "floor_complement": 1.0 - floor,
    }
    for k, v in enumerate(medians):
        row[f"pol_{k}"] = float(v) * scale_factor
    records.append(row)
medians_df = pd.DataFrame.from_records(records)

world = world.merge(medians_df, on="ISO_A3", how="left")
missing = world[world["best_policy"].isna()]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
world.boundary.plot(ax=ax, color="k", linewidth=0.4)
ax.set_xticks([])
ax.set_yticks([])
world.plot(ax=ax, column="floor_complement", cmap="Blues")

mpl.rcParams["hatch.color"] = "lightgrey"
missing.plot(ax=ax, facecolor="white", edgecolor="none", hatch="//")

fig.savefig("floor_complement.png")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 10))
world.boundary.plot(ax=ax, color="k", linewidth=0.4)
ax.set_xticks([])
ax.set_yticks([])

policy_codes = OXCGRT_COLMAP["custom"]
policy_colours = [OXCGRT_LOCATION_CMAP[i] for i in policy_codes]
policy_names = [MOB_LOCATION_NAME_MAP[i] for i in policy_codes]

cmap = mcolors.ListedColormap(policy_colours)
world.plot(ax=ax, column="best_policy", edgecolor="none", cmap=cmap)


handles = [Patch(facecolor=c, label=n) for n, c in zip(policy_names, policy_colours)]
ax.legend(handles=handles, loc="lower left", fontsize=16)

fig.savefig("best_policy.png")

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(20, 18))
flat_axes = axes.ravel()
missing = world[world["best_policy"].isna()]
for a, ax in enumerate(flat_axes):
    world.boundary.plot(ax=ax, color="k", linewidth=0.4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(MOB_LOCATION_NAME_MAP[policy_codes[a]], fontsize=22)

    world.plot(column=f"pol_{a}", cmap="Blues", ax=ax, legend=True, edgecolor="none")
    
    missing.plot(ax=ax, facecolor="white", edgecolor="none", hatch="//")
fig.tight_layout()
fig.savefig("component_effects.png")